# 25 · RAG 提示词与引文生成

> 好的 RAG prompt 是把“检索到的资料”转成“有据可依的答案”的最后开关，并让答案能**指出来源（Citation）**。

**本文件覆盖知识点**：Context Injection / System Prompt / Grounding Prompt / Citation Prompt / Don't-Know Prompt / Context Separation / Citation · Source Attribution

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. RAG Prompt 的组成

```text
[System]  只根据 Context 回答；没有答案就答"不知道"；不编造。
[User]    参考资料(每条带编号，与 metadata 关联):
          [1] 星云产品手册·第2页: 支持私有化部署
          [2] 星云产品手册·第3页: 套餐分三档
          
          问题: 它支持私有化吗？
[Answer]  支持[1]。产品提供公有云 SaaS 与私有化两种方式。
```

四大要素：
- **Context Injection**：把检索片段拼入，编号方便引用；
- **Grounding / Context Separation**：明确区分“资料”与“问题”，禁止混合；
- **Don't-Know Prompt**：资料不足就明说不知道（防幻觉的关键）；
- **Citation Prompt**：要求模型用 [来源N] 标注，N 对应真实 chunk。

In [2]:
# .env 配置
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def build_messages(question, contexts):
    """contexts: [(text, meta), ...] → 拼成带来源编号的 RAG 消息"""
    system = ('你是严谨的知识库助手。只依据参考资料作答，禁止编造；'
              '资料中找不到答案时回答"根据已有资料无法回答"；'
              '引用资料请在句末用 [来源N] 标注。')
    lines = []
    for i, (text, meta) in enumerate(contexts, 1):
        src = f"{meta.get('source','?')}·p{meta.get('page','?')}" if meta else '?'
        lines.append(f'[来源{i}]（{src}） {text}')
    return [{'role':'system','content':system},
            {'role':'user','content':'参考资料：\n' + '\n'.join(lines) + f'\n\n问题：{question}'}]

ctx = [('星云客服机器人支持公有云 SaaS 与私有化两种部署方式', {'source':'星云智能产品手册','page':2})]
for m in build_messages('支持私有化部署吗？', ctx):
    print(f'--- {m["role"]} ---')
    print(m['content'])

def ask(q, contexts):
    r = Generation.call(model='qwen-plus', messages=build_messages(q, contexts),
                        api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content

if API_KEY and '你的' not in API_KEY:
    print('\n回答:', ask('支持私有化部署吗？', ctx))
    # 反面验证：问一个资料里没有的，模型应守住“不知道”
    print('\n越界问题回答:', ask('产品在火星上能用吗？', ctx))

--- system ---
你是严谨的知识库助手。只依据参考资料作答，禁止编造；资料中找不到答案时回答"根据已有资料无法回答"；引用资料请在句末用 [来源N] 标注。
--- user ---
参考资料：
[来源1]（星云智能产品手册·p2） 星云客服机器人支持公有云 SaaS 与私有化两种部署方式

问题：支持私有化部署吗？

回答: 支持私有化部署 [来源1]。

越界问题回答: 根据已有资料无法回答。


In [3]:
# 知识点·真调说明：Citation Prompt —— 让模型按 system 的“句末 [来源N]”要求作答，再程序化校验引用是否真实
import re as _re
cite_srcs = [
    ('星云客服机器人支持公有云 SaaS 与私有化两种部署方式。', '星云产品手册·P2'),
    ('标准版 998 元/月，含 5 个坐席、自动应答与基础报表。', '星云产品手册·P3'),
    ('私有化部署需联系销售单独开通，一般 5~10 个工作日完成。', '星云产品手册·P4'),
]
ctx_lines = '\n'.join('[%d]（%s）%s' % (i + 1, meta, t) for i, (t, meta) in enumerate(cite_srcs))
print('① 拼进上下文的参考资料（带来源编号，[N] 对应第 N 条）:')
print(ctx_lines)
print()
out = _llm_live(
    prompt='参考资料：\n%s\n\n问题：星云客服机器人支持私有化部署吗？标准版月费多少、含哪些能力？' % ctx_lines,
    system='你是严谨的知识库问答助手。规则：1) 只依据上面参考资料作答，资料里没有就答“根据资料无法确认”，绝不编造；'
           '2) 凡论断来自某条资料，就在对应句末用 [来源N] 标注，N 必须指向上面的编号；'
           '3) 不要输出任何其它解释。',
    fallback='未配置 Key 的固定样例：\n'
             '支持[1]。星云客服机器人提供公有云 SaaS 与私有化两种部署方式[1]。\n'
             '标准版月费 998 元，含 5 个坐席、自动应答与基础报表[2]。',
    temperature=0.2,
)
if out is None:
    out = ('支持[1]。星云客服机器人提供公有云 SaaS 与私有化两种部署方式[1]。\n'
           '标准版月费 998 元，含 5 个坐席、自动应答与基础报表[2]。')
    print('（以上为固定样例；下面用样例演示“引用校验”）')
print()
print('② 程序化校验引用（正则抓出 [来源N]，看编号是否落在资料范围）:')
nums = [int(x) for x in _re.findall(r'\[(?:来源)?(\d+)\]', out)]
if not nums:
    print('  没检出任何 [来源N] 标注 —— 说明引用约束没被遵守，需要把要求写得更硬')
else:
    uniq = sorted(set(nums))
    ok = all(1 <= n <= len(cite_srcs) for n in uniq)
    print('  检出引用编号:', uniq, '  全部落在 1..%d 内: %s' % (len(cite_srcs), ok))
print('→ 句末 [N] 让每一句都能对应回真实 chunk（来源即第 9 课存的 metadata），前端据此渲染可点击的引用卡片；'
      '“模型乱标来源”的逐条校验兜底在第 26 课。')

① 拼进上下文的参考资料（带来源编号，[N] 对应第 N 条）:
[1]（星云产品手册·P2）星云客服机器人支持公有云 SaaS 与私有化两种部署方式。
[2]（星云产品手册·P3）标准版 998 元/月，含 5 个坐席、自动应答与基础报表。
[3]（星云产品手册·P4）私有化部署需联系销售单独开通，一般 5~10 个工作日完成。

—— 模型实时输出 ——
星云客服机器人支持私有化部署 [1]。标准版月费998元，含5个坐席、自动应答与基础报表 [2]。

② 程序化校验引用（正则抓出 [来源N]，看编号是否落在资料范围）:
  检出引用编号: [1, 2]   全部落在 1..3 内: True
→ 句末 [N] 让每一句都能对应回真实 chunk（来源即第 9 课存的 metadata），前端据此渲染可点击的引用卡片；“模型乱标来源”的逐条校验兜底在第 26 课。


## 2. Citation 从哪来到哪去

- **来源**：就是第 9 课存的 chunk metadata（document_id/page/source）；
- **生成**：prompt 里让模型在 [来源N] 引用；N 对应检索列表第 N 条；
- **验证**：可选 LLM 逐条校验“这句话真的来自来源N”（防模型乱标，见第 26 课）；
- **呈现**：前端把 [N] 渲染成卡片，点击跳到原文。

## 小结

- RAG prompt 四要素：**注入+分离+不知道+引用**；
- Citation 让答案可溯源、可校验、可信任；
- “资料不足就答不知道”是缓解幻觉的第一道防线。